In [ ]:
# load the dataset
from pathlib import Path

from battery_aging.data_preprocessing import load_dataset

Dataset = "Stanford"
data_folder = Path("..", "Battery_life_Dataset", Dataset)

# n_files -- Number of files to load (default is "all" to load all files)
datasets = load_dataset(data_folder, n_files="all", seed=42)

print([(d["cell_id"], d["discharge_protocol"]) for d in datasets])

In [ ]:
# reduce the number of cycles in the dataset to speed up the analysis based 
# on the specified step size and the cycles to keep

from battery_aging.data_preprocessing import reduce_cycles

keep_cycles=[5, 10, 15, 30]
step = 20

for dataset in datasets:
    dataset["cycle_data"] = reduce_cycles(dataset["cycle_data"], step=step, keep_cycles=keep_cycles)

# Print the number of cycles in each dataset and the first 10 cycle numbers
for dataset in datasets:
    print(
        dataset["cell_id"],
        len(dataset["cycle_data"]),
        [c["cycle_number"] for c in dataset["cycle_data"][:10]]
    )

Stanford_Nova_Regular_191 60 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_192 54 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_193 50 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_194 56 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_195 58 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_196 54 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_198 50 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_199 58 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_200 59 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_201 53 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_202 58 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_203 49 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_204 49 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_205 49 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_206 60 [1, 2, 3, 4, 5, 10, 15, 20, 30, 40]
Stanford_Nova_Regular_207

In [ ]:
# First round of feature engineering
# list of features to be extracted from the cycle data:
# SOH, I_mean, I_std, V_mean, V_std, charge_duration, discharge_duration
from battery_aging.feature_engineering import feature_engineering_1

features_df = feature_engineering_1(datasets)



In [ ]:
for d in datasets:
    print(d["cell_id"], len(d["cycle_data"]))

print(len(datasets[0]["cycle_data"]))

Stanford_Nova_Regular_191 60
Stanford_Nova_Regular_192 54
Stanford_Nova_Regular_193 50
Stanford_Nova_Regular_194 56
Stanford_Nova_Regular_195 58
Stanford_Nova_Regular_196 54
Stanford_Nova_Regular_198 50
Stanford_Nova_Regular_199 58
Stanford_Nova_Regular_200 59
Stanford_Nova_Regular_201 53
Stanford_Nova_Regular_202 58
Stanford_Nova_Regular_203 49
Stanford_Nova_Regular_204 49
Stanford_Nova_Regular_205 49
Stanford_Nova_Regular_206 60
Stanford_Nova_Regular_207 63
Stanford_Nova_Regular_208 46
Stanford_Nova_Regular_209 52
Stanford_Nova_Regular_210 47
Stanford_Nova_Regular_211 57
Stanford_Nova_Regular_212 48
Stanford_Nova_Regular_213 59
Stanford_Nova_Regular_214 54
Stanford_Nova_Regular_215 62
Stanford_Nova_Regular_216 54
Stanford_Nova_Regular_217 60
Stanford_Nova_Regular_219 49
Stanford_Nova_Regular_220 58
Stanford_Nova_Regular_221 50
Stanford_Nova_Regular_222 49
Stanford_Nova_Regular_223 59
Stanford_Nova_Regular_224 79
Stanford_Nova_Regular_225 60
Stanford_Nova_Regular_226 64
Stanford_Nova_

In [ ]:
# Second round of feature engineering
# list of features to be extracted from the cycle data:
# V_range, V_slope_mean, V_curvature, V_n_peaks, RMSE_V, AreaDiff_V, Corr_V, MaxDev_V, SlopeRMSE_V
from battery_aging.feature_engineering import feature_engineering_2

features_df_2 = feature_engineering_2(datasets)
print(features_df_2.columns)
print(features_df_2.head())

features_df = features_df.merge(
    features_df_2,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'V_range', 'V_slope_mean', 'V_curvature',
       'V_n_peaks', 'RMSE_V', 'AreaDiff_V', 'Corr_V', 'MaxDev_V',
       'SlopeRMSE_V'],
      dtype='str')
                     cell_id  cycle_number   V_range  V_slope_mean  \
0  Stanford_Nova_Regular_191             1  1.400244      0.001404   
1  Stanford_Nova_Regular_191             2  1.400244      0.001404   
2  Stanford_Nova_Regular_191             3  1.400168      0.001404   
3  Stanford_Nova_Regular_191             4  1.400244      0.001404   
4  Stanford_Nova_Regular_191             5  1.400167      0.001404   

   V_curvature  V_n_peaks    RMSE_V  AreaDiff_V    Corr_V  MaxDev_V  \
0     0.000379         11  0.000841    0.000428  0.999997  0.010576   
1     0.000373          9  0.000000    0.000000  1.000000  0.000000   
2     0.000368         12  0.000597    0.000294  0.999999  0.007031   
3     0.000364         12  0.001095    0.000556  0.999996  0.012542   
4     0.000256         11  0.064429    0

In [ ]:
# third round of feature engineering: DTW features based on current curves

from battery_aging.feature_engineering import feature_engineering_dtw_current

dtw_df = feature_engineering_dtw_current(datasets)

print(dtw_df.columns)
print(dtw_df.head())

features_df = features_df.merge(
    dtw_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'DTW_I'], dtype='str')
                     cell_id  cycle_number     DTW_I
0  Stanford_Nova_Regular_191             1  0.046376
1  Stanford_Nova_Regular_191             2  0.000000
2  Stanford_Nova_Regular_191             3  0.038445
3  Stanford_Nova_Regular_191             4  0.044773
4  Stanford_Nova_Regular_191             5  0.378790


In [ ]:
# fourth round of feature engineering: DTW features based on voltage curves

from battery_aging.feature_engineering import feature_engineering_dtw_voltage

dtw_v_df = feature_engineering_dtw_voltage(datasets)

print(dtw_v_df.columns)
print(dtw_v_df.head())

features_df = features_df.merge(
    dtw_v_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

Index(['cell_id', 'cycle_number', 'DTW_V'], dtype='str')
                     cell_id  cycle_number     DTW_V
0  Stanford_Nova_Regular_191             1  0.065362
1  Stanford_Nova_Regular_191             2  0.000000
2  Stanford_Nova_Regular_191             3  0.048579
3  Stanford_Nova_Regular_191             4  0.074361
4  Stanford_Nova_Regular_191             5  3.218301


In [ ]:
# Display the columns of the merged DataFrame and a sample of the new features

print(features_df.columns.tolist())
# print(features_df [
#     [
#         "V_range",
#         "V_slope_mean",
#         "V_curvature",
#         "V_n_peaks"
#     ]
# ].head())
print(features_df.head())

['cell_id', 'cycle_number', 'SOH', 'I_mean', 'I_std', 'charge_duration', 'discharge_duration', 'V_mean', 'V_std', 'V_range', 'V_slope_mean', 'V_curvature', 'V_n_peaks', 'RMSE_V', 'AreaDiff_V', 'Corr_V', 'MaxDev_V', 'SlopeRMSE_V', 'DTW_I', 'DTW_V']
                     cell_id  cycle_number       SOH    I_mean     I_std  \
0  Stanford_Nova_Regular_191             1  1.000355  0.002598  0.193819   
1  Stanford_Nova_Regular_191             2  1.000000  0.002665  0.193852   
2  Stanford_Nova_Regular_191             3  0.999869  0.002707  0.193887   
3  Stanford_Nova_Regular_191             4  0.999600  0.002730  0.193907   
4  Stanford_Nova_Regular_191             5  1.001134  0.004111  0.194839   

   charge_duration  discharge_duration    V_mean     V_std   V_range  \
0      4435.116211         4878.810059  3.848261  0.324460  1.400244   
1      4429.852539         4877.081055  3.847929  0.324621  1.400244   
2      4426.253906         4876.439453  3.847701  0.324773  1.400168   
3      

In [ ]:
import pickle
from pathlib import Path

# Zielordner
output_dir = Path("..", "engineered_data")
output_dir.mkdir(exist_ok=True)

# Dateiname
filename = output_dir / f"processed_battery_features_{Dataset}.pkl"

# Speichern
with open(filename, "wb") as f:
    pickle.dump(features_df, f)

print(f"Saved to: {filename.resolve()}")

Saved to: C:\Users\Christoph\Documents\VS_Studio_Projekte\VS_Studio_Projekte\engineered_data\processed_battery_features_Stanford.pkl
